
# SIH 2026 PS33 — Farmer-to-Buyer Vehicle Route Optimization

This Google Colab notebook builds a working V1 route optimization engine for the SIH PS33 concept.

It covers:
- sample farmers, buyers, vehicles
- farmer-supply to buyer-demand matching
- road distance/time matrix using OSRM, with a Haversine fallback
- Google OR-Tools multi-vehicle pickup & delivery routing
- vehicle capacity constraints
- pickup-before-delivery constraints
- pickup/delivery time windows
- maximum route duration
- route KPIs
- Folium map visualization
- API-ready JSON output

The clean team pipeline is:

Demand Forecasting → Order/Demand Aggregation → Farmer/Buyer Matching → Shipment List → Route Optimizer → Optimized Routes → Dashboard


In [ ]:

# CELL 1 — Install dependencies

!pip -q install ortools pandas numpy requests folium matplotlib

print("Dependencies installed.")


In [ ]:

# CELL 2 — Imports and configuration

import math
import json
import requests
import numpy as np
import pandas as pd
import folium
import matplotlib.pyplot as plt

from ortools.constraint_solver import pywrapcp, routing_enums_pb2

USE_OSRM = True
OSRM_BASE_URL = "https://router.project-osrm.org"
ROUTING_PROFILE = "driving"

SOLVER_TIME_LIMIT_SECONDS = 10

COST_PER_KM = 12.0
COST_PER_HOUR = 80.0
FIXED_VEHICLE_COST = 300.0

MINUTES_PER_DAY = 24 * 60

print("Ready.")



## 1. Sample input data

Replace these tables later with your team's real data/API.

A farmer is a pickup location. A buyer is a delivery location. A vehicle starts and ends at the depot. A shipment connects one pickup with one delivery and carries a quantity in kg.


In [ ]:

# CELL 3 — Sample data

DEPOT = {
    "id": "D0",
    "name": "Pune Collection Hub",
    "lat": 18.5204,
    "lon": 73.8567,
}

farmers = pd.DataFrame([
    {
        "farmer_id": "F01", "name": "Farmer A",
        "lat": 18.5074, "lon": 73.8077,
        "product": "Tomato", "supply_kg": 180,
        "pickup_start": "06:00", "pickup_end": "10:00",
        "service_min": 15,
    },
    {
        "farmer_id": "F02", "name": "Farmer B",
        "lat": 18.5913, "lon": 73.7389,
        "product": "Tomato", "supply_kg": 220,
        "pickup_start": "06:00", "pickup_end": "10:30",
        "service_min": 15,
    },
    {
        "farmer_id": "F03", "name": "Farmer C",
        "lat": 18.6298, "lon": 73.7997,
        "product": "Tomato", "supply_kg": 160,
        "pickup_start": "06:30", "pickup_end": "11:00",
        "service_min": 15,
    },
    {
        "farmer_id": "F04", "name": "Farmer D",
        "lat": 18.4529, "lon": 73.8553,
        "product": "Tomato", "supply_kg": 140,
        "pickup_start": "06:00", "pickup_end": "09:30",
        "service_min": 15,
    },
])

buyers = pd.DataFrame([
    {
        "buyer_id": "B01", "name": "Wholesale Buyer 1",
        "lat": 18.5679, "lon": 73.9143,
        "product": "Tomato", "demand_kg": 250,
        "delivery_start": "09:00", "delivery_end": "13:00",
        "service_min": 20,
    },
    {
        "buyer_id": "B02", "name": "Wholesale Buyer 2",
        "lat": 18.6420, "lon": 73.7610,
        "product": "Tomato", "demand_kg": 220,
        "delivery_start": "09:00", "delivery_end": "14:00",
        "service_min": 20,
    },
    {
        "buyer_id": "B03", "name": "Retail Aggregator",
        "lat": 18.4770, "lon": 73.8900,
        "product": "Tomato", "demand_kg": 120,
        "delivery_start": "08:30", "delivery_end": "12:00",
        "service_min": 20,
    },
])

vehicles = pd.DataFrame([
    {
        "vehicle_id": "V01",
        "capacity_kg": 350,
        "max_route_min": 540,
        "fixed_cost": FIXED_VEHICLE_COST,
    },
    {
        "vehicle_id": "V02",
        "capacity_kg": 350,
        "max_route_min": 540,
        "fixed_cost": FIXED_VEHICLE_COST,
    },
    {
        "vehicle_id": "V03",
        "capacity_kg": 300,
        "max_route_min": 540,
        "fixed_cost": FIXED_VEHICLE_COST,
    },
])

display(farmers)
display(buyers)
display(vehicles)


In [ ]:

# CELL 4 — Helper functions

def time_to_minutes(hhmm):
    h, m = map(int, hhmm.split(":"))
    return h * 60 + m


def parse_window(start, end):
    return [time_to_minutes(start), time_to_minutes(end)]


def haversine_km(lat1, lon1, lat2, lon2):
    R = 6371.0
    p1 = math.radians(lat1)
    p2 = math.radians(lat2)
    dlat = math.radians(lat2 - lat1)
    dlon = math.radians(lon2 - lon1)

    a = (
        math.sin(dlat / 2) ** 2
        + math.cos(p1) * math.cos(p2) * math.sin(dlon / 2) ** 2
    )
    return 2 * R * math.asin(math.sqrt(a))


print("Helpers loaded.")



## 2. Supply → demand matching

For V1, this uses a simple greedy matching layer:
1. Buyers are sorted by delivery deadline.
2. Compatible farmer supply is assigned to buyer demand.
3. A farmer can be split across buyers.
4. Each allocation becomes one shipment.

Later, replace this layer with a transportation/min-cost-flow model if you want globally better supply-demand assignment.


In [ ]:

# CELL 5 — Greedy farmer-to-buyer shipment creation

def build_shipments(farmers_df, buyers_df):
    farmers_work = farmers_df.copy()
    farmers_work["remaining_kg"] = farmers_work["supply_kg"].astype(float)

    buyers_work = buyers_df.sort_values(
        by="delivery_end",
        key=lambda s: s.map(time_to_minutes)
    )

    shipments = []
    shipment_counter = 1

    for _, buyer in buyers_work.iterrows():
        remaining_demand = float(buyer["demand_kg"])

        compatible = farmers_work[
            (farmers_work["product"] == buyer["product"])
            & (farmers_work["remaining_kg"] > 0)
        ].sort_values(
            by="pickup_end",
            key=lambda s: s.map(time_to_minutes)
        )

        for farmer_idx, farmer in compatible.iterrows():
            if remaining_demand <= 0:
                break

            alloc = min(
                float(farmer["remaining_kg"]),
                remaining_demand
            )

            if alloc <= 0:
                continue

            shipments.append({
                "shipment_id": f"S{shipment_counter:02d}",
                "product": buyer["product"],
                "quantity_kg": alloc,

                "farmer_id": farmer["farmer_id"],
                "farmer_name": farmer["name"],
                "pickup_lat": farmer["lat"],
                "pickup_lon": farmer["lon"],
                "pickup_start": farmer["pickup_start"],
                "pickup_end": farmer["pickup_end"],
                "pickup_service_min": int(farmer["service_min"]),

                "buyer_id": buyer["buyer_id"],
                "buyer_name": buyer["name"],
                "delivery_lat": buyer["lat"],
                "delivery_lon": buyer["lon"],
                "delivery_start": buyer["delivery_start"],
                "delivery_end": buyer["delivery_end"],
                "delivery_service_min": int(buyer["service_min"]),
            })

            farmers_work.loc[farmer_idx, "remaining_kg"] -= alloc
            remaining_demand -= alloc
            shipment_counter += 1

        if remaining_demand > 0.001:
            raise ValueError(
                f"Demand for {buyer['buyer_id']} is not fully covered. "
                f"Shortage = {remaining_demand:.1f} kg"
            )

    return pd.DataFrame(shipments)


shipments = build_shipments(farmers, buyers)

display(shipments)

print("Total farmer supply:", farmers["supply_kg"].sum(), "kg")
print("Total buyer demand:", buyers["demand_kg"].sum(), "kg")
print("Total routed quantity:", shipments["quantity_kg"].sum(), "kg")


In [ ]:

# CELL 6 — Build routing nodes and pickup/delivery pairs

def build_nodes_and_pairs(shipments_df, depot):
    nodes = [{
        "node_id": 0,
        "node_key": "DEPOT",
        "type": "DEPOT",
        "label": depot["name"],
        "lat": depot["lat"],
        "lon": depot["lon"],
        "load_change_kg": 0,
        "time_window": [0, MINUTES_PER_DAY],
        "service_min": 0,
        "shipment_id": None,
    }]

    pickup_node_by_shipment = {}
    delivery_node_by_shipment = {}
    next_node = 1

    for _, s in shipments_df.iterrows():

        pickup_node = next_node
        next_node += 1

        nodes.append({
            "node_id": pickup_node,
            "node_key": f"{s['shipment_id']}_P",
            "type": "PICKUP",
            "label": f"{s['farmer_name']} ({s['farmer_id']})",
            "lat": s["pickup_lat"],
            "lon": s["pickup_lon"],
            "load_change_kg": int(round(s["quantity_kg"])),
            "time_window": parse_window(s["pickup_start"], s["pickup_end"]),
            "service_min": int(s["pickup_service_min"]),
            "shipment_id": s["shipment_id"],
        })

        delivery_node = next_node
        next_node += 1

        nodes.append({
            "node_id": delivery_node,
            "node_key": f"{s['shipment_id']}_D",
            "type": "DELIVERY",
            "label": f"{s['buyer_name']} ({s['buyer_id']})",
            "lat": s["delivery_lat"],
            "lon": s["delivery_lon"],
            "load_change_kg": -int(round(s["quantity_kg"])),
            "time_window": parse_window(s["delivery_start"], s["delivery_end"]),
            "service_min": int(s["delivery_service_min"]),
            "shipment_id": s["shipment_id"],
        })

        pickup_node_by_shipment[s["shipment_id"]] = pickup_node
        delivery_node_by_shipment[s["shipment_id"]] = delivery_node

    pairs = [
        (pickup_node_by_shipment[sid], delivery_node_by_shipment[sid])
        for sid in pickup_node_by_shipment
    ]

    return pd.DataFrame(nodes), pairs


nodes_df, pickup_delivery_pairs = build_nodes_and_pairs(shipments, DEPOT)

display(nodes_df)
print("Pickup-delivery pairs:", pickup_delivery_pairs)



## 3. Distance/time matrix

OSRM's Table service can return road distances and travel durations between all supplied coordinates. The code below uses it for the demo and falls back to Haversine distance + assumed speed if the public service cannot be reached.


In [ ]:

# CELL 7 — OSRM / Haversine matrix

def build_osrm_matrix(coords, base_url=OSRM_BASE_URL):
    coord_string = ";".join(
        f"{lon},{lat}" for lat, lon in coords
    )

    url = (
        f"{base_url}/table/v1/{ROUTING_PROFILE}/"
        f"{coord_string}?annotations=distance,duration"
    )

    response = requests.get(
        url,
        timeout=30,
        headers={"User-Agent": "SIH-PS33-Route-Optimizer/1.0"}
    )
    response.raise_for_status()

    data = response.json()

    if data.get("code") != "Ok":
        raise RuntimeError(f"OSRM returned: {data.get('code')}")

    distances = np.array(data["distances"], dtype=float) / 1000.0
    durations = np.array(data["durations"], dtype=float) / 60.0

    if not np.isfinite(distances).all() or not np.isfinite(durations).all():
        raise RuntimeError("OSRM returned missing/invalid matrix values.")

    return distances, durations


def build_haversine_matrix(coords, assumed_speed_kmph=35.0):
    n = len(coords)
    distance = np.zeros((n, n), dtype=float)
    duration = np.zeros((n, n), dtype=float)

    for i in range(n):
        for j in range(n):
            km = haversine_km(
                coords[i][0], coords[i][1],
                coords[j][0], coords[j][1]
            )
            distance[i, j] = km
            duration[i, j] = (km / assumed_speed_kmph) * 60.0

    return distance, duration


coords = list(zip(nodes_df["lat"], nodes_df["lon"]))

if USE_OSRM:
    try:
        distance_matrix_km, duration_matrix_min = build_osrm_matrix(coords)
        MATRIX_SOURCE = "OSRM road network"
    except Exception as e:
        print("OSRM failed:", repr(e))
        print("Using Haversine fallback.")
        distance_matrix_km, duration_matrix_min = build_haversine_matrix(coords)
        MATRIX_SOURCE = "Haversine fallback"
else:
    distance_matrix_km, duration_matrix_min = build_haversine_matrix(coords)
    MATRIX_SOURCE = "Haversine fallback"

print("Matrix source:", MATRIX_SOURCE)

display(pd.DataFrame(
    np.round(distance_matrix_km, 2),
    index=nodes_df["node_key"],
    columns=nodes_df["node_key"]
))

display(pd.DataFrame(
    np.round(duration_matrix_min, 1),
    index=nodes_df["node_key"],
    columns=nodes_df["node_key"]
))



## 4. The actual OR-Tools model

Objective:
- minimize road distance
- minimize travel time
- penalize use of vehicles with a fixed cost

Constraints:
- vehicle capacity
- same vehicle for pickup and delivery
- pickup before delivery
- pickup/delivery time windows
- route duration limit


In [ ]:

# CELL 8 — OR-Tools solver

def solve_route_optimization(
    nodes_df,
    pickup_delivery_pairs,
    distance_matrix_km,
    duration_matrix_min,
    vehicles_df,
    depot_node=0,
    time_limit_seconds=10,
):
    num_nodes = len(nodes_df)
    num_vehicles = len(vehicles_df)

    capacities = vehicles_df["capacity_kg"].astype(int).tolist()
    max_route_minutes = vehicles_df["max_route_min"].astype(int).tolist()

    manager = pywrapcp.RoutingIndexManager(
        num_nodes,
        num_vehicles,
        depot_node
    )

    routing = pywrapcp.RoutingModel(manager)

    # -------------------------
    # Distance objective
    # -------------------------
    def distance_callback(from_index, to_index):
        from_node = manager.IndexToNode(from_index)
        to_node = manager.IndexToNode(to_index)
        return int(round(distance_matrix_km[from_node, to_node] * 1000))

    distance_callback_index = routing.RegisterTransitCallback(
        distance_callback
    )

    routing.SetArcCostEvaluatorOfAllVehicles(
        distance_callback_index
    )

    # -------------------------
    # Travel time
    # -------------------------
    def duration_callback(from_index, to_index):
        from_node = manager.IndexToNode(from_index)
        to_node = manager.IndexToNode(to_index)
        return int(round(duration_matrix_min[from_node, to_node]))

    duration_callback_index = routing.RegisterTransitCallback(
        duration_callback
    )

    # -------------------------
    # Capacity
    # -------------------------
    load_changes = nodes_df["load_change_kg"].astype(int).tolist()

    def demand_callback(from_index):
        from_node = manager.IndexToNode(from_index)
        return load_changes[from_node]

    demand_callback_index = routing.RegisterUnaryTransitCallback(
        demand_callback
    )

    routing.AddDimensionWithVehicleCapacity(
        demand_callback_index,
        0,
        capacities,
        True,
        "Capacity"
    )

    capacity_dimension = routing.GetDimensionOrDie("Capacity")

    # -------------------------
    # Time dimension
    # -------------------------
    service_times = nodes_df["service_min"].astype(int).tolist()

    def total_time_callback(from_index, to_index):
        from_node = manager.IndexToNode(from_index)
        to_node = manager.IndexToNode(to_index)

        travel = duration_matrix_min[from_node, to_node]
        service = service_times[from_node]

        return int(round(travel + service))

    total_time_callback_index = routing.RegisterTransitCallback(
        total_time_callback
    )

    max_route_duration = max(max_route_minutes)

    routing.AddDimension(
        total_time_callback_index,
        max_route_duration,
        max_route_duration,
        False,
        "Time"
    )

    time_dimension = routing.GetDimensionOrDie("Time")

    time_dimension.SetGlobalSpanCostCoefficient(10)

    # -------------------------
    # Node time windows
    # -------------------------
    for _, row in nodes_df.iterrows():
        index = manager.NodeToIndex(int(row["node_id"]))
        start, end = row["time_window"]

        time_dimension.CumulVar(index).SetRange(
            int(start),
            int(end)
        )

    # -------------------------
    # Vehicle settings
    # -------------------------
    for vehicle_id in range(num_vehicles):
        start_index = routing.Start(vehicle_id)
        end_index = routing.End(vehicle_id)

        time_dimension.CumulVar(start_index).SetRange(
            0, max_route_minutes[vehicle_id]
        )

        time_dimension.CumulVar(end_index).SetRange(
            0, max_route_minutes[vehicle_id]
        )

        fixed_cost = int(round(
            vehicles_df.iloc[vehicle_id]["fixed_cost"]
        ))

        routing.SetFixedCostOfVehicle(
            fixed_cost,
            vehicle_id
        )

    # -------------------------
    # Pickup & delivery
    # -------------------------
    for pickup_node, delivery_node in pickup_delivery_pairs:
        pickup_index = manager.NodeToIndex(pickup_node)
        delivery_index = manager.NodeToIndex(delivery_node)

        routing.AddPickupAndDelivery(
            pickup_index,
            delivery_index
        )

        # Same vehicle.
        routing.solver().Add(
            routing.VehicleVar(pickup_index)
            == routing.VehicleVar(delivery_index)
        )

        # Pickup before delivery.
        routing.solver().Add(
            time_dimension.CumulVar(pickup_index)
            <= time_dimension.CumulVar(delivery_index)
        )

    # -------------------------
    # Search parameters
    # -------------------------
    search_parameters = pywrapcp.DefaultRoutingSearchParameters()

    search_parameters.first_solution_strategy = (
        routing_enums_pb2.FirstSolutionStrategy.PATH_CHEAPEST_ARC
    )

    search_parameters.local_search_metaheuristic = (
        routing_enums_pb2.LocalSearchMetaheuristic.GUIDED_LOCAL_SEARCH
    )

    search_parameters.time_limit.seconds = int(time_limit_seconds)

    solution = routing.SolveWithParameters(search_parameters)

    return manager, routing, solution, capacity_dimension, time_dimension


manager, routing, solution, capacity_dimension, time_dimension = solve_route_optimization(
    nodes_df=nodes_df,
    pickup_delivery_pairs=pickup_delivery_pairs,
    distance_matrix_km=distance_matrix_km,
    duration_matrix_min=duration_matrix_min,
    vehicles_df=vehicles,
    time_limit_seconds=SOLVER_TIME_LIMIT_SECONDS,
)

if solution is None:
    raise RuntimeError(
        "No feasible route found. Increase vehicle capacity, "
        "add vehicles, or widen time windows."
    )

print("Solution found.")
print("Solver objective:", solution.ObjectiveValue())


In [ ]:

# CELL 9 — Extract readable routes

def extract_solution(
    manager,
    routing,
    solution,
    nodes_df,
    vehicles_df,
    distance_matrix_km,
    duration_matrix_min,
    capacity_dimension,
    time_dimension,
):
    node_lookup = nodes_df.set_index("node_id").to_dict("index")

    all_routes = []
    summaries = []

    for vehicle_id in range(len(vehicles_df)):
        index = routing.Start(vehicle_id)

        # Skip unused vehicles.
        if routing.IsEnd(solution.Value(routing.NextVar(index))):
            continue

        vehicle = vehicles_df.iloc[vehicle_id]
        route = []
        route_distance = 0.0
        route_travel_min = 0.0
        previous_index = None

        while not routing.IsEnd(index):
            node_id = manager.IndexToNode(index)
            node = node_lookup[node_id]

            arrival_min = solution.Min(
                time_dimension.CumulVar(index)
            )

            load_after = solution.Value(
                capacity_dimension.CumulVar(index)
            )

            route.append({
                "vehicle_id": vehicle["vehicle_id"],
                "sequence": len(route) + 1,
                "node_id": node_id,
                "node_key": node["node_key"],
                "type": node["type"],
                "label": node["label"],
                "shipment_id": node["shipment_id"],
                "arrival_min": arrival_min,
                "arrival_hhmm": f"{int(arrival_min // 60):02d}:{int(arrival_min % 60):02d}",
                "load_after_visit_kg": load_after,
                "lat": node["lat"],
                "lon": node["lon"],
            })

            if previous_index is not None:
                prev = manager.IndexToNode(previous_index)

                route_distance += distance_matrix_km[prev, node_id]
                route_travel_min += duration_matrix_min[prev, node_id]

            previous_index = index
            index = solution.Value(routing.NextVar(index))

        # Final depot/end node.
        end_node_id = manager.IndexToNode(index)
        end_node = node_lookup[end_node_id]

        arrival_min = solution.Min(
            time_dimension.CumulVar(index)
        )

        load_after = solution.Value(
            capacity_dimension.CumulVar(index)
        )

        route.append({
            "vehicle_id": vehicle["vehicle_id"],
            "sequence": len(route) + 1,
            "node_id": end_node_id,
            "node_key": end_node["node_key"],
            "type": end_node["type"],
            "label": end_node["label"],
            "shipment_id": end_node["shipment_id"],
            "arrival_min": arrival_min,
            "arrival_hhmm": f"{int(arrival_min // 60):02d}:{int(arrival_min % 60):02d}",
            "load_after_visit_kg": load_after,
            "lat": end_node["lat"],
            "lon": end_node["lon"],
        })

        if previous_index is not None:
            prev = manager.IndexToNode(previous_index)
            route_distance += distance_matrix_km[prev, end_node_id]
            route_travel_min += duration_matrix_min[prev, end_node_id]

        capacity = float(vehicle["capacity_kg"])
        max_load = max(x["load_after_visit_kg"] for x in route)

        estimated_cost = (
            route_distance * COST_PER_KM
            + (route_travel_min / 60.0) * COST_PER_HOUR
            + float(vehicle["fixed_cost"])
        )

        summaries.append({
            "vehicle_id": vehicle["vehicle_id"],
            "distance_km": route_distance,
            "travel_time_min": route_travel_min,
            "max_load_kg": max_load,
            "capacity_kg": capacity,
            "utilization_pct": max_load / capacity * 100,
            "estimated_cost_inr": estimated_cost,
        })

        all_routes.extend(route)

    return pd.DataFrame(all_routes), pd.DataFrame(summaries)


routes_df, summary_df = extract_solution(
    manager,
    routing,
    solution,
    nodes_df,
    vehicles,
    distance_matrix_km,
    duration_matrix_min,
    capacity_dimension,
    time_dimension
)

print("OPTIMIZED ROUTES")
display(routes_df)

print("ROUTE SUMMARY")
display(summary_df)


In [ ]:

# CELL 10 — KPIs

total_distance = summary_df["distance_km"].sum()
total_travel_hours = summary_df["travel_time_min"].sum() / 60.0
total_cost = summary_df["estimated_cost_inr"].sum()
vehicles_used = len(summary_df)
total_goods = shipments["quantity_kg"].sum()

weighted_utilization = (
    summary_df["max_load_kg"].sum()
    / summary_df["capacity_kg"].sum()
    * 100
)

kpis = {
    "vehicles_used": vehicles_used,
    "total_distance_km": round(total_distance, 2),
    "total_travel_hours": round(total_travel_hours, 2),
    "total_goods_kg": round(total_goods, 2),
    "capacity_utilization_pct": round(weighted_utilization, 2),
    "estimated_transport_cost_inr": round(total_cost, 2),
    "cost_per_kg_inr": round(total_cost / total_goods, 2),
}

print(json.dumps(kpis, indent=2))


In [ ]:

# CELL 11 — Folium visualization

center_lat = nodes_df["lat"].mean()
center_lon = nodes_df["lon"].mean()

route_map = folium.Map(
    location=[center_lat, center_lon],
    zoom_start=11
)

folium.Marker(
    [DEPOT["lat"], DEPOT["lon"]],
    tooltip="DEPOT",
    popup=DEPOT["name"],
    icon=folium.Icon(icon="home")
).add_to(route_map)

for _, row in routes_df.iterrows():
    if row["type"] == "PICKUP":
        marker_color = "green"
    elif row["type"] == "DELIVERY":
        marker_color = "red"
    else:
        marker_color = "blue"

    folium.Marker(
        [row["lat"], row["lon"]],
        tooltip=f"{row['vehicle_id']} — stop {row['sequence']}",
        popup=(
            f"<b>{row['label']}</b><br>"
            f"Type: {row['type']}<br>"
            f"Shipment: {row['shipment_id']}<br>"
            f"Arrival: {row['arrival_hhmm']}<br>"
            f"Load after visit: {row['load_after_visit_kg']} kg"
        ),
        icon=folium.Icon(color=marker_color)
    ).add_to(route_map)

for vehicle_id, group in routes_df.groupby("vehicle_id"):
    coords_route = [
        [lat, lon]
        for lat, lon in zip(group["lat"], group["lon"])
    ]

    folium.PolyLine(
        coords_route,
        tooltip=f"Route {vehicle_id}",
        weight=5
    ).add_to(route_map)

route_map


In [ ]:

# CELL 12 — Simple baseline comparison

def baseline_sequential_distance(nodes_df, distance_matrix_km):
    ordered_nodes = nodes_df.sort_values("node_id")["node_id"].tolist()

    distance = 0.0

    for a, b in zip(ordered_nodes[:-1], ordered_nodes[1:]):
        distance += distance_matrix_km[a, b]

    if ordered_nodes[-1] != 0:
        distance += distance_matrix_km[ordered_nodes[-1], 0]

    return distance


baseline_distance = baseline_sequential_distance(
    nodes_df,
    distance_matrix_km
)

distance_saved_pct = (
    (baseline_distance - total_distance)
    / baseline_distance
    * 100
)

print(f"Baseline distance:  {baseline_distance:.2f} km")
print(f"Optimized distance: {total_distance:.2f} km")
print(f"Reduction:          {distance_saved_pct:.2f}%")


In [ ]:

# CELL 13 — API-ready JSON output

def build_api_response(summary_df, routes_df):
    routes = []

    for vehicle_id, group in routes_df.groupby("vehicle_id"):
        summary = summary_df[
            summary_df["vehicle_id"] == vehicle_id
        ].iloc[0]

        stops = []

        for _, row in group.iterrows():
            stops.append({
                "sequence": int(row["sequence"]),
                "node": row["node_key"],
                "type": row["type"],
                "label": row["label"],
                "shipment_id": row["shipment_id"],
                "arrival": row["arrival_hhmm"],
                "load_after_visit_kg": float(row["load_after_visit_kg"]),
                "latitude": float(row["lat"]),
                "longitude": float(row["lon"]),
            })

        routes.append({
            "vehicle_id": vehicle_id,
            "distance_km": round(float(summary["distance_km"]), 2),
            "travel_time_min": round(float(summary["travel_time_min"]), 1),
            "capacity_kg": round(float(summary["capacity_kg"]), 1),
            "max_load_kg": round(float(summary["max_load_kg"]), 1),
            "utilization_pct": round(float(summary["utilization_pct"]), 2),
            "estimated_cost_inr": round(float(summary["estimated_cost_inr"]), 2),
            "stops": stops,
        })

    return {
        "status": "success",
        "matrix_source": MATRIX_SOURCE,
        "kpis": kpis,
        "routes": routes,
    }


api_response = build_api_response(summary_df, routes_df)
print(json.dumps(api_response, indent=2))



# 13. What to replace for the real SIH project

1. Replace the sample farmer/buyer tables with your team's real database/API.
2. Replace the greedy matching layer with your final matching logic.
3. Feed the demand-forecasting output into buyer/order demand.
4. Cache geocoding and distance matrices.
5. Use actual road geometry for each optimized leg.
6. Add perishability and maximum transit-time constraints.
7. Add heterogeneous vehicle types if required.
8. Add dynamic re-optimization for cancellations/new orders.
9. Wrap `solve_route_optimization()` inside a FastAPI endpoint such as `POST /optimize-route`.
10. Return routes + ETA + distance + cost + utilization to the frontend.

The key separation is:

**Forecasting predicts WHAT needs to move.  
Matching decides WHICH supply can satisfy demand.  
Your routing optimizer decides HOW to move it.**
